<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">

# The Data Scientist
## Book 2 · Python Data Analysis, Visualization, and Storytelling
### Notebook 03 · pandas Deep Dive
© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br> The Python Quants GmbH | https://tpq.io<br>

https://thedatascientist.dev | https://linktr.ee/dyjh

### Why this notebook exists
The book needs a bridge between first-table habits and the more realistic workflows
that use multiple files. This notebook gives that bridge a small, repeatable shape.

This cell makes the notebook portable. It finds the book root locally, or clones
`tdscode` in Colab, so the later paths and imports work without manual edits.

This cell makes the notebook portable. It finds the book root locally, or clones
`tdscode` in Colab, so the later paths and imports work without manual edits.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "2_data"
REPO_URL = "https://github.com/yhilpisch/tdscode"

cwd = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in (cwd, *cwd.parents):
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")
    if not repo_root.exists():
        # Clone the companion repo once in Colab.
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
    BOOK_ROOT = repo_root / BOOK_NAME

os.chdir(BOOK_ROOT)

# Make the book root and the helper folder importable.
for path in (BOOK_ROOT, BOOK_ROOT / "code"):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

requirements = BOOK_ROOT / "requirements.txt"
if requirements.exists() and "google.colab" in sys.modules:
    # Keep Colab aligned with the book dependencies.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

print("Book root:", BOOK_ROOT)


In [ ]:
from pathlib import Path

import pandas as pd

data_dir = BOOK_ROOT / 'data'
artifacts_dir = BOOK_ROOT / 'artifacts'
artifacts_dir.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(data_dir / 'orders.csv')
customers = pd.read_csv(data_dir / 'customers.csv')

print(orders.head())
print()
print(customers.head())


This cell reads the orders and customers CSV files separately so the learner can see
the tables before a join.

In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders_with_customers = orders.merge(customers, on='customer_id', how='left')

print(orders_with_customers.head())
print()
print(orders_with_customers.dtypes)


This cell merges orders with customers and checks the data types. The joined table is
the base for the later summaries.

In [ ]:
country_summary = (
    orders_with_customers
    .groupby('country')['amount']
    .agg(total_amount='sum', average_amount='mean', order_count='size')
    .sort_values('total_amount', ascending=False)
)

category_summary = (
    orders_with_customers
    .pivot_table(
        index='country',
        columns='category',
        values='amount',
        aggfunc='sum',
        fill_value=0,
    )
)

print(country_summary)
print()
print(category_summary)


This cell summarizes order amounts by country and category. Grouping and pivoting
make the multi-table data easier to compare.

In [ ]:
clean_orders = (
    orders_with_customers
    .drop_duplicates()
    .assign(
        amount_eur=lambda df: df['amount'].round(2),
        order_month=lambda df: df['order_date'].dt.to_period('M').astype(str),
    )
)

print(clean_orders[['order_id', 'country', 'amount_eur', 'order_month']])


This cell removes duplicates, adds a month column, and keeps a clean amount field.
The result is the tidy table the later report can use.

In [ ]:
def format_amount(value: float) -> str:
    return f'{value:,.2f}'

amount_preview = clean_orders['amount'].head().apply(format_amount)

print('formatted amounts:')
print(amount_preview.tolist())

output_path = data_dir / 'orders_clean.csv'
clean_orders.to_csv(output_path, index=False)
print(output_path.resolve())


This cell formats amounts for display and writes the cleaned order table to disk. It
shows that analysis often ends with a reusable file.

In [ ]:
summary_path = artifacts_dir / 'pandas_deep_dive_summary.txt'
summary_path.write_text(
    'pandas deep dive summary\n'
    f'Rows: {len(clean_orders):,}\n'
    f'Countries: {", ".join(country_summary.index.tolist())}\n'
    f'Categories: {", ".join(sorted(clean_orders["category"].unique()))}\n'
)

print(summary_path.resolve())


<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">